# Spline versus GIN/FiLM validation across radius

**Role:** Analysis.

Compare completed exclusive-fate spline models with a saved reference model across neighborhood radius/GNN depth. Load models and their exact saved validation inputs; never train or refit preprocessing. Verify matching outer train/validation organoid membership, graph edges, node targets and marker panel before comparison. Differences in input encoding, head globals and objectives are reported explicitly: equal folds do not imply identical model inputs.

Figures show total and optional anatomical-region MSE, plus paired per-organoid MSE differences relative to the reference at the same radius. Each model uses its own fitted target transform and baseline offsets. Error bands are one SEM across organoids after averaging repeated seeds, not uncertainty from retraining. The center-only spline is fitted at radius zero and reused as a horizontal comparator, not retrained at other radii.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/models/gnn.py').is_file())  # Repository root.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Resolve data and saved-run paths relative to this directory.

## Analysis settings

Select the completed spline run and the reference family, width, seed and optional masking rate. If REFERENCE_RUN is None, use the source run recorded by spline training. All saved spline folds are included. Reference mode requires the same outer folds; mismatches raise instead of silently intersecting cohorts. Optional regions are rebuilt from source annotations using one common definition for both models.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from src.artifacts.paths import resolve_training_run
from src.artifacts.runs import AnalysisRun
from src.artifacts.bundle import graph_membership
from src.data.marker_exclusivity import exclusive_markers
from src.analysis.metrics.evaluation import prediction_table, organoid_mse_summary, regional_mse
from src.analysis.spatial.regions import graph_regions

# Saved runs
SPLINE_RUN = None  # Name under training_results/fate_spline_training; None requires exactly one completed run.
REFERENCE_RUN = None  # Explicit project-relative run, or inherit the source recorded by spline training.

# Model selection
REFERENCE_NAME = 'film'  # Reference family, e.g. gin or film.
REFERENCE_WIDTH = 128  # Reference hidden width; None when not applicable.
REFERENCE_SEED = 42  # One reference initialization, across every matching fold.
REFERENCE_RATE = 0.0  # Zero-masking FiLM control from the Ndependent run.
SPLINE_SEEDS = None  # None includes every saved spline inner-split seed.
RADII = [0, 1, 2, 3, 4]  # Spline radii to display, including the radius-zero center model.
REFERENCE_DEPTHS = None  # None uses available reference depths within RADII; the default run has only depth 2.

# Evaluation and display
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Device for reference inference; spline inference uses CPU internally.
EVALUATE_REGIONS = True  # Include qualified crypt, neck and villus panels.
ALLOW_ENCODING_DIFFERENCE = False  # Permit a nonexclusive reference, with its input differences explicitly reported.
REGION_SETTINGS = dict(
    crypt_max=.8,  # Crypt body upper distance boundary.
    neck_max=1.2,  # Qualified neck upper distance boundary.
    neck_config=dict(
        minimum_crypt_cells=10,  # Minimum cells supporting a crypt profile.
        minimum_relative_depth=.05,  # Relative circumference trough threshold.
        plateau_max_range=.10,  # Relative flat-section range limit.
        plateau_max_slope=.25,  # Flat-section slope limit.
        plateau_exit_rise=.10,  # Required rise beyond the flat section.
    ),
)

## Select complete model grids and verify the comparison

Use one reference architecture and initialization at each available reference depth. The default zero-masking FiLM run contains only depth 2: absolute MSE still shows spline radii 0–4, while the reference and paired differences are shown only at depth 2. No missing reference depths are interpolated or trained. For every fold, compare the actual loaded memberships and targets, not only fold numbers. The exclusive spline inputs must equal the reference fates after the recorded exclusivity rules; extra reference channels may only be an observed (zero) missingness flag. Anatomical labels and measured curvature never enter the spline input features.

In [ ]:
spline_dir = resolve_training_run(ROOT, 'fate_spline_training', SPLINE_RUN)
spline_run = AnalysisRun(spline_dir)
reference_path = REFERENCE_RUN or spline_run.settings.get('reference_run')
if reference_path is None:
    raise ValueError('Choose REFERENCE_RUN; this spline run was trained on a fresh cohort.')
reference_run = AnalysisRun(ROOT / reference_path)
rows = spline_run.records[spline_run.records.depth.isin(RADII)].copy()
if SPLINE_SEEDS is not None:
    rows = rows[rows.seed.isin(SPLINE_SEEDS)]
if rows.empty:
    raise ValueError('No requested spline models.')
folds = {s['fold'] for s in json.loads((spline_dir / 'splits.json').read_text())}
expected_seeds = set(spline_run.settings['seeds'] if SPLINE_SEEDS is None else SPLINE_SEEDS)
for _, part in rows.groupby(['name','depth']):
    if part.duplicated(['fold','seed']).any() or set(zip(part.fold, part.seed)) != {(f,s) for f in folds for s in expected_seeds}:
        raise ValueError('Incomplete spline fold/seed grid.')
ref = reference_run.records
ref = ref[(ref.name == REFERENCE_NAME) & (ref.seed == REFERENCE_SEED) &
          (ref.subset == 'all') & (ref.signal == 'intact') & ref.depth.isin(RADII)]
if REFERENCE_WIDTH is not None:
    ref = ref[ref.hidden_dim == REFERENCE_WIDTH]
if 'rate' in ref and ref.rate.notna().any():
    if REFERENCE_RATE is None:
        raise ValueError('Choose REFERENCE_RATE explicitly.')
    ref = ref[np.isclose(ref.rate, REFERENCE_RATE)]
reference_depths = sorted(ref.depth.unique().tolist()) if REFERENCE_DEPTHS is None else list(REFERENCE_DEPTHS)
if not reference_depths or len(set(reference_depths)) != len(reference_depths) or not set(reference_depths) <= set(RADII):
    raise ValueError('Choose nonempty, distinct reference depths within RADII and present in the reference run.')
ref = ref[ref.depth.isin(reference_depths)]
if ref.duplicated(['fold','depth']).any() or set(zip(ref.fold, ref.depth)) != {(f,d) for f in folds for d in reference_depths}:
    raise ValueError('Reference needs exactly one checkpoint per selected reference depth and spline fold.')
print('Reference depths available for comparison:', reference_depths)
OUTPUT_DIR = spline_dir / 'analysis' / 'radius_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audit = []
for fold in sorted(folds):
    left = spline_run.select(rows[rows.fold == fold].iloc[0].key)
    for record in ref[ref.fold == fold].itertuples():
        right = reference_run.select(record.key)
        if left['marker_names'] != right['marker_names']:
            raise ValueError('Marker names/order differ between runs.')
        for role in ('train', 'val'):
            lg = {str(g.organoid_str):g for g in left['groups'][role]}
            rg = {str(g.organoid_str):g for g in right['groups'][role]}
            if lg.keys() != rg.keys():
                raise ValueError(f'Different {role} membership in fold {fold}.')
            for oid, g in lg.items():
                other = rg[oid]
                if not torch.equal(g.edge_index.cpu(), other.edge_index.cpu()) or len(g.x) != len(other.x):
                    raise ValueError(f'{oid}: graph topology/node ordering differs.')
                y1 = np.asarray(left['transform'].inverse(g.y.cpu().numpy())).reshape(-1) + np.asarray(left['baseline_offsets'][oid]).reshape(-1)
                y2 = np.asarray(right['transform'].inverse(other.y.cpu().numpy())).reshape(-1) + np.asarray(right['baseline_offsets'][oid]).reshape(-1)
                np.testing.assert_allclose(y1, y2, rtol=2e-5, atol=1e-8, err_msg=f'{oid}: physical targets differ')
                panel = other.x[:, :len(left['marker_names'])].cpu().numpy()
                if other.x.shape[1] > len(left['marker_names']) and torch.any(other.x[:, len(left['marker_names']):] != 0):
                    raise ValueError('Reference inputs contain missing fates.')
                np.testing.assert_array_equal(g.x.cpu().numpy(), exclusive_markers(panel, left['marker_names']))
                if not ALLOW_ENCODING_DIFFERENCE and not np.array_equal(g.x.cpu().numpy(), panel):
                    raise ValueError('Reference is not fate-exclusive. Enable ALLOW_ENCODING_DIFFERENCE for a predictive benchmark.')
        audit.append(dict(fold=fold, reference_key=record.key, depth=record.depth,
            reference_globals=right.get('global_features'), reference_exclusive=reference_run.settings.get('exclusive_markers'),
            spline_inputs='exclusive fates, shell fractions, log N', matched_membership=True, matched_targets=True))
print('Verified identical memberships, topology and physical targets for every reference depth/fold.')
print('Reference head globals:', audit[0]['reference_globals'], '| exclusive:', audit[0]['reference_exclusive'])
print('Spline loss: mean organoid physical-curvature MSE; reference objective:', reference_run.settings.get('loss_name', 'see source settings'))
(OUTPUT_DIR / 'comparison_settings.json').write_text(json.dumps(dict(spline_run=str(spline_dir), reference_run=str(reference_run.directory),
    spline_keys=rows.key.tolist(), reference_keys=ref.key.tolist(), reference_depths=reference_depths, regions=REGION_SETTINGS, audit=audit), indent=2))

## Evaluate held-out predictions and source-defined regions

Run each checkpoint on its own saved validation inputs. Aggregate squared errors within organoids before combining folds. Region annotations are shared across methods, including explicit categories for unavailable/unqualified crypts. Save all scores so plots remain reviewable independently of the notebook kernel.

In [ ]:
parts, annotations = [], {}
for source_name, run, selected_rows in [('Spline', spline_run, rows), ('Reference', reference_run, ref)]:
    for record in selected_rows.to_dict('records'):
        chosen = run.select(record['key'], device='cpu' if source_name == 'Spline' else DEVICE)
        predictions = prediction_table(chosen, device='cpu' if source_name == 'Spline' else DEVICE)
        total = predictions.groupby('organoid_str').agg(mse=('squared_error','mean'), baseline_mse=('baseline_squared_error','mean'), n_cells=('node','size')).reset_index()
        total['region'] = 'total'
        tables = [total]
        if EVALUATE_REGIONS:
            for g in chosen['groups']['val']:
                if g.organoid_str not in annotations:
                    annotations[g.organoid_str] = graph_regions(chosen['raw_graphs'][g.organoid_str], ROOT/'training_data'/spline_run.settings['dataset'], **REGION_SETTINGS)
            table = pd.concat([annotations[g.organoid_str] for g in chosen['groups']['val']], ignore_index=True)
            tables.append(regional_mse(predictions, table))
        for table in tables:
            for key in ('key', 'name', 'depth', 'fold', 'seed'):
                table[key] = record[key]
            table['source'] = source_name
            parts.append(table)
score_table = pd.concat(parts, ignore_index=True)
score_table.to_csv(OUTPUT_DIR / 'validation_mse.csv', index=False)
if annotations:
    pd.concat(annotations.values(), ignore_index=True).to_csv(OUTPUT_DIR / 'node_regions.csv.gz', index=False)

## Compare MSE and paired differences across radii

Upper panels show absolute MSE and each run's baseline. Lower panels show spline minus reference MSE on exactly matched fold/organoid/region rows, then average repeated spline seeds within organoid. Negative differences favor the spline. The center-only curve is repeated visually at each requested radius; it remains one radius-zero model. SEM bands are not hypothesis tests and do not include retraining uncertainty.

In [ ]:
plot_scores = score_table.copy()
centers = plot_scores[(plot_scores.source == 'Spline') & (plot_scores.name == 'spline_center')]
plot_scores = plot_scores[~((plot_scores.source == 'Spline') & (plot_scores.name == 'spline_center'))]
plot_scores = pd.concat([plot_scores, *[centers.assign(depth=d) for d in RADII]], ignore_index=True)
regions = ['total'] + (['crypt', 'neck', 'villus'] if EVALUATE_REGIONS else [])
fig, axes = plt.subplots(2, len(regions), figsize=(5*len(regions), 8), squeeze=False, sharex='col')
for col, region in enumerate(regions):
    frame = plot_scores[plot_scores.region == region]
    for (source, name), part in frame.groupby(['source','name']):
        curve = organoid_mse_summary(part, 'depth')
        line, = axes[0,col].plot(curve.depth, curve['mean'], 'o-', label=f'{source}: {name}')
        axes[0,col].fill_between(curve.depth, curve['mean']-curve['sem'], curve['mean']+curve['sem'], color=line.get_color(), alpha=.15)
    for source, part in frame.groupby('source'):
        curve = organoid_mse_summary(part, 'depth', value='baseline_mse')
        line, = axes[0,col].plot(curve.depth, curve['mean'], '--', label=f'{source} baseline')
        axes[0,col].fill_between(curve.depth, curve['mean']-curve['sem'], curve['mean']+curve['sem'], color=line.get_color(), alpha=.08)
    reference_scores = frame[frame.source == 'Reference'][['fold','organoid_str','depth','mse']].rename(columns={'mse':'reference_mse'})
    for name, part in frame[frame.source == 'Spline'].groupby('name'):
        paired = part[part.depth.isin(reference_depths)].merge(reference_scores, on=['fold','organoid_str','depth'], how='left', validate='many_to_one')
        if paired.reference_mse.isna().any():
            raise ValueError('Missing matched reference score.')
        paired['difference'] = paired.mse - paired.reference_mse
        curve = organoid_mse_summary(paired, 'depth', value='difference')
        line, = axes[1,col].plot(curve.depth, curve['mean'], 'o-', label=name)
        axes[1,col].fill_between(curve.depth, curve['mean']-curve['sem'], curve['mean']+curve['sem'], color=line.get_color(), alpha=.15)
    axes[0,col].set(title=region.capitalize(), ylabel='Validation MSE (curvature²)')
    axes[1,col].axhline(0, color='black', linewidth=.8)
    axes[1,col].set(xlabel='Radius / GNN depth', ylabel='Spline − reference MSE', xticks=RADII)
    axes[1,col].set_title('Paired comparisons at reference depths: ' + ', '.join(map(str, reference_depths)))
    for ax in axes[:,col]:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=7)
        else:
            ax.text(.5, .5, 'No qualified validation cells', ha='center', transform=ax.transAxes)
fig.suptitle('Equal-organoid means ± SEM; center-only model reused across radii')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'mse_comparison.png', dpi=160, bbox_inches='tight')
plt.show()